# Few-shot IGT with Bolmo 1B (Ainu / Jejueo / Tsez)

Same two settings as the zero-shot notebook (`zero_shot_bolmo.ipynb`), both **one model call**:
- **all-in-one**: predict transliteration and gloss together
- **step-by-step**: the same prompt asks for Step 1 (transliteration), then Step 2 (gloss based on Step 1)

The only difference: the prompt first shows `SHOTS` solved examples from the **train** split (gold transliteration and gloss). The examples are fixed (the same for every test row, chosen with seed 42, see `few_shot` in `configs/zero_shot.json`), and no example text occurs in the dev or test split. Logic: `zero_shot/run_zero_shot.py --shots`.

Before running: **Runtime -> Change runtime type -> T4 GPU**.

## 1. Install (once per runtime), then Runtime -> Restart session

In [ ]:
!pip install "transformers==4.57.3" "xlstm==2.0.4" datasets pandas accelerate

## 2. Get the code and check the environment

In [ ]:
import os, sys, importlib, importlib.util
BRANCH = "main"
REPO = "/content/Korean-IGT-Bolmo"
if not os.path.isdir(REPO):
    !git clone https://github.com/JingyiShao-Gina/Korean-IGT-Bolmo.git {REPO}
!git -C {REPO} fetch origin
!git -C {REPO} checkout {BRANCH}
!git -C {REPO} pull origin {BRANCH}
os.chdir(REPO)
sys.path.insert(0, REPO + "/zero_shot")
!git log --oneline -1

import transformers
print("transformers:", transformers.__version__, "(need 4.57.3)")
print("xlstm installed:", importlib.util.find_spec("xlstm") is not None)

## 3. Choose the language and the number of examples, and preview the prompts (no model needed)

In [ ]:
LANG = "jejueo"   # "ainu", "jejueo" or "tsez"
SHOTS = 3         # number of train examples in the prompt

import json, pandas as pd
import run_zero_shot as rz
importlib.reload(rz)
cfg = json.load(open("configs/zero_shot.json"))
F = cfg["few_shot"]
L = cfg["languages"][LANG]
manifest = rz.ROOT / L["dataset_manifest"]
name = L["language_name"]
eval_texts = [r["text"] for sp in ("dev", "test") for r in rz.load_rows(sp, None, manifest)]
examples = rz.select_examples(rz.load_rows("train", None, manifest), SHOTS, F["seed"], eval_texts,
                              F["max_text_chars"], F["max_gloss_chars"], F["min_text_words"])
block = rz.examples_block(examples, name)
T = {k: rz.add_examples(rz.fill_language(rz.load_prompt(cfg["prompts"][k]), name), block, name)
     for k in ("all_in_one", "step_by_step")}
ex = rz.load_rows("dev", 1, manifest)[0]
for key in ("all_in_one", "step_by_step"):
    print(f"----- {key} -----")
    print(rz.render(T[key], original_script=ex["text"], translation=ex["translation"]))
print("----- gold -----")
print(ex["transcription"], "|", ex["glosses"])

## 4. Smoke test: 10 dev rows, both settings

This only checks the pipeline. It is **not** a result. The examples are the same as in the test run.

In [ ]:
pd.set_option("display.max_colwidth", 80)
smoke = rz.main(["--lang", LANG, "--split", "dev", "--limit", "10", "--shots", str(SHOTS)])
for p in smoke:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print()
    print("=====", p.name, "=====")
    print(df.parse_status.value_counts().to_dict(), "| truncated:", int(df.truncated.sum()),
          "| seconds:", meta["runtime_seconds"], "| mean new tokens:", round(df.n_new_tokens.mean()))
    display(df[["gold_transliteration", "pred_transliteration", "gold_gloss", "pred_gloss", "parse_status"]])
    display(df[["raw_output"]])

## 5. Full run on the test split

Run this only after the smoke test looks right. Do not change prompts after looking at test outputs.

Time on a T4: the 10-row Jejueo dev smoke test took about 2.5 min per setting (zero-shot: 45 s), mostly because more outputs run into repetition loops. Expect roughly **2 hours per setting** on the test split. To be safe, run one setting per session: set `STRATEGY = "all_in_one"`, download the zip (section 6), then run again with `STRATEGY = "step_by_step"`. Each setting writes its own file.

In [ ]:
STRATEGY = "all_in_one"   # "all_in_one", then "step_by_step" ("both" runs the two in a row)
full = rz.main(["--lang", LANG, "--split", "test", "--shots", str(SHOTS), "--strategy", STRATEGY])
for p in full:
    df = pd.read_csv(p, keep_default_na=False)
    meta = json.load(open(p.with_suffix(".meta.json")))
    print(p.name, "| rows:", len(df), "|", df.parse_status.value_counts().to_dict(),
          "| truncated:", int(df.truncated.sum()), "| minutes:", round(meta["runtime_seconds"] / 60, 1))

## 6. Download the results

In [ ]:
out_dir = F["output_dir"].format(lang=LANG)
!zip -r /content/few_shot_{LANG}_{SHOTS}shot_results.zip {out_dir}
from google.colab import files
files.download(f"/content/few_shot_{LANG}_{SHOTS}shot_results.zip")